## 1. INSTALLS, IMPORTS, AND PARAMETERS

### --- 1. Installs ---

In [29]:
!pip install geopandas geojson overpass alphashape torch

### --- 2. Imports ---

In [30]:
import pandas as pd
import json
import geojson
import overpass
import csv
import os
import logging
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
from shapely.geometry import Point, Polygon
import requests
import time
from google.colab import drive
import sys
import ast
from alphashape import alphashape
import geopandas as gpd
import os
from collections import defaultdict
import os.path
from tqdm import tqdm
import numpy as np
import pickle
import random
import json
from itertools import permutations
import time as time_module
import networkx as nx
from datetime import datetime
import geopy.distance
import heapq
import math
import heapq
import pickle
import os

# (New imports for PyTorch / RL)
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import copy
from collections import namedtuple, deque

### --- 3. Global Parameters ---

In [31]:
state_name = "Ohio"
city_name = "Columbus"
mini = True

### 1. Setup

In [32]:
# --- 4. Connect to Drive ---
try:
    drive.mount('/content/gdrive/', force_remount=True)
    print("Google Drive mounted successfully.")
except Exception as e:
    print(f"Error mounting Google Drive: {e}")

Mounted at /content/gdrive/
Google Drive mounted successfully.


In [33]:
# --- 5. Set Directories ---
try:
    personal_dir = "./gdrive/MyDrive/UMST_DeliverAI/Delivery_Data/"
    data_dir = f"{personal_dir}{city_name}_mini - RL Delivery Data" if mini else f"{personal_dir}{city_name} - RL Delivery Data"

    # We will save our new MADDPG models and results in a new folder
    output_dir = data_dir + "/UMST Graph/maddpg_baseline"

    os.makedirs(output_dir, exist_ok=True)
    print(f"Data directory: {data_dir}")
    print(f"Output directory: {output_dir}")

    # See directory content
    data = os.listdir(data_dir)
    print(f"Files in data_dir: {data}")

except Exception as e:
    print(f"Error setting up directories. Is your drive mounted and the path correct?")
    print(f"Personal Dir Path: {personal_dir}")
    print(f"Data Dir Path: {data_dir}")

Data directory: ./gdrive/MyDrive/UMST_DeliverAI/Delivery_Data/Columbus_mini - RL Delivery Data
Output directory: ./gdrive/MyDrive/UMST_DeliverAI/Delivery_Data/Columbus_mini - RL Delivery Data/UMST Graph/maddpg_baseline
Files in data_dir: ['avg_hotspot_data.json', 'results_rangewise', 'UMST Graph', 'results_randomized', 'results_fixed_buffer', 'results_multiply', 'Processed Location Data', 'Q Tables', 'Images', 'simulation_results', 'Original Location Data', 'gh_cache', 'Census Data', 'Hotspot Data', 'Deliveries']


# 2. RE-USABLE DATA & ENVIRONMENT CLASSES (THE "WORLD")
 This is to keep the deliveries same as they were even when operating on two different methods. This follows the same structure as used in the file `V7_02_order_bundling.ipynb`

#### 1. Delivery Class


In [34]:
INTERVALS = [900, 1200, 1500, 1800, 2700, 3600, 5400, 7200, 10800, 14400]

In [35]:
class Delivery:
    """
    Represents a single delivery request.
    (This is the simplified version from your notebook)
    """
    id_counter = 0

    def __init__(self, start_node, end_node, start_time, time_tolerance_factor,
                 shortest_path=None):
        # Identity
        self.id = Delivery.id_counter
        Delivery.id_counter += 1

        # Route (immutable)
        self.start_node = start_node
        self.end_node = end_node
        self.shortest_path = shortest_path if shortest_path else [start_node, end_node]

        # Time constraints
        self.start_time = start_time
        self.TIME_TOLERANCE_FACTOR = time_tolerance_factor
        self.time_limit = None  # Set after initialization

        # Current state (mutable)
        self.current_node = start_node
        self.path_index = 0
        self.in_transition = False
        self.time_till_next_node = 0
        self.wait_time_remaining = 0 # Used for heuristic baseline, but good to keep

        # Completion tracking
        self.completed = False
        self.successful = False
        self.end_time = None

        # Statistics
        self.distance_traveled = 0
        self.actual_path = [start_node]
        self.num_vehicle_changes = 0 # We'll re-purpose this for the RL agent
        self.times_bundled = 0

        # Bundle reference (managed by Bundle class or RL Env)
        self.current_bundle_id = None

    def get_next_node(self):
        """Get next node in planned path."""
        if self.path_index < len(self.shortest_path) - 1:
            return self.shortest_path[self.path_index + 1]
        return None

    def move_to_next_node(self, distance, travel_time):
        """Start transition to next node."""
        self.in_transition = True
        self.time_till_next_node = travel_time
        self.distance_traveled += distance

    def arrive_at_node(self, node):
        """Complete arrival at node."""
        self.in_transition = False
        self.current_node = node
        self.actual_path.append(node)
        self.path_index += 1

        # CRITICAL: Check if reached destination
        if self.current_node == self.end_node:
            self.completed = True
            # Note: successful status set later

    def reset(self):
        """Reset to initial state."""
        self.current_node = self.start_node
        self.path_index = 0
        self.in_transition = False
        self.time_till_next_node = 0
        self.wait_time_remaining = 0
        self.completed = False
        self.successful = False
        self.end_time = None
        self.distance_traveled = 0
        self.actual_path = [self.start_node]
        self.num_vehicle_changes = 0
        self.current_bundle_id = None

    def __repr__(self):
        status = "✅" if self.completed else ("🚗" if self.in_transition else "⏳")
        bundle = f"[B{self.current_bundle_id}]" if self.current_bundle_id else ""
        return f"D{self.id}:{self.start_node}→{self.end_node}|{status}{bundle}@N{self.current_node}"

#### 2. Graph & Pathfinding Helpers

In [36]:
def build_adjacency_matrix(graph: nx.Graph, tract_to_index, dist_attr='distance', time_attr='time'):
    """
    Create adjacency dict for direct edges only using integer indices.
    Each node maps to a list of (neighbor_index, distance, time_seconds)
    """
    adjacency = {}

    for u, v, data in graph.edges(data=True):
        u_idx = tract_to_index[u]
        v_idx = tract_to_index[v]
        dist = float(data.get(dist_attr, 0))
        time = float(data.get(time_attr, 0)) * 60  # minutes -> seconds
        adjacency.setdefault(u_idx, []).append((v_idx, dist, time))
        adjacency.setdefault(v_idx, []).append((u_idx, dist, time))

    num_edges = sum(len(v) for v in adjacency.values()) // 2
    print(f"✓ Adjacency built: {len(adjacency)} nodes with {num_edges} edges")
    return adjacency


def astar_shortest_path(adjacency, start, goal, node_positions=None, weight_type='time'):
    """
    Compute shortest path from start → goal using A*.
    - adjacency: dict[index] -> list of (neighbor_index, dist, time)
    - node_positions: dict[index] -> (lat, lon) for heuristic
    - weight_type: 'time' or 'distance'
    Returns: (total_cost, path_indices)
    """
    def heuristic(u, v):
        if node_positions is None:
            return 0
        (x1, y1), (x2, y2) = node_positions[u], node_positions[v]
        return math.hypot(x1 - x2, y1 - y2)

    frontier = [(0, start)]
    came_from = {start: None}
    cost_so_far = {start: 0}

    while frontier:
        _, current = heapq.heappop(frontier)
        if current == goal:
            break

        if adjacency.get(current) is None:
            continue # Node has no outgoing edges

        for neighbor, dist, time in adjacency.get(current, []):
            weight = time if weight_type == 'time' else dist
            new_cost = cost_so_far[current] + weight

            if neighbor not in cost_so_far or new_cost < cost_so_far[neighbor]:
                cost_so_far[neighbor] = new_cost
                priority = new_cost + heuristic(neighbor, goal)
                heapq.heappush(frontier, (priority, neighbor))
                came_from[neighbor] = current

    # Reconstruct path
    if goal not in came_from:
        return float('inf'), []

    path = []
    node = goal
    while node is not None:
        path.append(node)
        node = came_from[node]
    path.reverse()

    return cost_so_far[goal], path

#### 3. DeliveryList Class (The Task Generator)

(This is UNCHANGED from the original file)
This is CRUCIAL for ensuring we test both models on the *exact same* deliveries.

In [37]:
class DeliveryList:
    """Manages collection of deliveries with shortest path calculation."""

    def __init__(self, max_delivery_time, load, time_tolerance_factor,
                 hours=1, peaks=[0.25, 0.75], sigma=10, num_hotspots=50,
                 adjacency_matrix=None, node_positions=None):

        self.load = load
        self.hours = hours
        self.peaks = peaks
        self.sigma = sigma
        self.time_tolerance_factor = time_tolerance_factor
        self.max_delivery_time = max_delivery_time
        self.num_hotspots = num_hotspots
        self.adjacency_matrix = adjacency_matrix
        self.node_positions = node_positions

        if adjacency_matrix is None or node_positions is None:
            raise ValueError("adjacency_matrix and node_positions are required!")

        print("📊 Generating temporal distribution...")
        self.distribution, self.loads = self.generate_distribution()

        print("🗺️  Pre-computing all shortest paths...")
        self.precomputed_paths = {}  # (start, end) -> (travel_time, path)
        self.valid_pairs = []  # List of (start, end) pairs within time limit
        self.precompute_all_paths()

        print("📦 Generating deliveries with pre-computed paths...")
        self.deliveries = self.generate_deliveries()

        print("⏱️  Calculating time limits...")
        self.calculate_time_limits()

        print(f"✅ Generated {len(self.deliveries)} deliveries")

    def generate_distribution(self):
        mu_list = [peak * 60 * self.hours for peak in self.peaks]
        x = np.linspace(0, 60 * self.hours, 60 * self.hours)
        y_combined = np.zeros_like(x)
        for mu in mu_list:
            y_combined += (1 / (self.sigma * np.sqrt(2 * np.pi))) * \
                         np.exp(-0.5 * ((x - mu) / self.sigma) ** 2)
        loads = [int((self.load * (self.sigma * np.sqrt(2 * np.pi))) * y)
                for y in y_combined]
        return y_combined, loads

    def precompute_all_paths(self):
        valid_nodes = list(range(self.num_hotspots))
        total_pairs = len(valid_nodes) * (len(valid_nodes) - 1)

        with tqdm(total=total_pairs, desc="Pre-computing paths") as pbar:
            for start in valid_nodes:
                for end in valid_nodes:
                    if start != end:
                        travel_time, path = astar_shortest_path(
                            adjacency=self.adjacency_matrix,
                            start=start,
                            goal=end,
                            node_positions=self.node_positions,
                            weight_type='time'
                        )
                        self.precomputed_paths[(start, end)] = (travel_time, path)
                        if travel_time <= self.max_delivery_time:
                            self.valid_pairs.append((start, end))
                        pbar.update(1)

        print(f"✓ Pre-computed {len(self.precomputed_paths)} shortest paths")
        print(f"✓ Found {len(self.valid_pairs)} valid delivery pairs (within {self.max_delivery_time}s limit)")

        if not self.valid_pairs:
            raise ValueError(f"No valid delivery pairs found within {self.max_delivery_time}s time limit!")

    def calculate_shortest_path(self, start, end):
        if start == end: return [start]
        if (start, end) not in self.precomputed_paths:
            print(f"⚠️ WARNING: Path ({start}, {end}) not found in pre-computed paths!")
            return [start, end]
        travel_time, path = self.precomputed_paths[(start, end)]
        return path

    def get_travel_time(self, start, end):
        if start == end: return 0
        if (start, end) not in self.precomputed_paths:
            print(f"⚠️ WARNING: Path ({start}, {end}) not found in pre-computed paths!")
            return float('inf')
        travel_time, path = self.precomputed_paths[(start, end)]
        return travel_time

    def calculate_time_limits(self):
        for delivery in self.deliveries:
            travel_time = self.get_travel_time(delivery.start_node, delivery.end_node)
            required_time = travel_time * delivery.TIME_TOLERANCE_FACTOR
            for interval in INTERVALS:
                if required_time <= interval:
                    delivery.time_limit = interval
                    break
            else:
                delivery.time_limit = float('inf')

    def generate_deliveries(self):
        deliveries = []
        total_deliveries = sum(self.loads)
        if not self.valid_pairs:
            raise ValueError("No valid delivery pairs available!")

        with tqdm(total=total_deliveries, desc="Creating deliveries") as pbar:
            for j in range(60 * self.hours):
                for _ in range(self.loads[j]):
                    start, end = random.choice(self.valid_pairs)
                    travel_time, shortest_path = self.precomputed_paths[(start, end)]
                    time = random.randint(j * 60, (j + 1) * 60)
                    delivery = Delivery(
                        start, end, time,
                        time_tolerance_factor=self.time_tolerance_factor,
                        shortest_path=shortest_path
                    )
                    deliveries.append(delivery)
                    pbar.update(1)
        return deliveries

    def reset_deliveries(self):
        """Reset all deliveries to their initial state."""
        Delivery.id_counter = 0 # Reset the global ID counter
        for d in self.deliveries:
            d.reset()
            # Re-assign IDs to be consistent
            d.id = Delivery.id_counter
            Delivery.id_counter += 1

#### 4. Load Graph Data

(This is UNCHANGED from the original file)

In [38]:
print("\n[1] Loading graph and census data...")
try:
    census_df = gpd.read_file(data_dir + "/Census Data/census_tract_data.geojson")
    num_hotspots = len(census_df.index)
    print(f"    ✓ Census tracts loaded: {num_hotspots} tracts (hotspots)")

    # --- CORRECTED FILE PATHS ---
    # Added .xml to the end based on your screenshot
    umst_path = data_dir + "/UMST Graph/graphs/umst_graph.graphml.xml"
    mst_path = data_dir + "/UMST Graph/graphs/mst_graph.graphml.xml"
    hotspot_path = data_dir + "/UMST Graph/graphs/gh_hotspot_graph.graphml.xml"

    umst_graph = nx.read_graphml(umst_path)
    print(f"    ✓ UMST Graph loaded: {umst_graph.number_of_nodes()} nodes, {umst_graph.number_of_edges()} edges")

    # --- 5. Build Adjacency & Position Dictionaries ---
    # (This section is the same as before)
    print("\n[2] Building node mappings and adjacency...")
    nodes = sorted(umst_graph.nodes())
    index_to_tract = {idx: node for idx, node in enumerate(nodes)}
    tract_to_index = {node: idx for idx, node in enumerate(nodes)}

    adjacency_matrix = build_adjacency_matrix(umst_graph, tract_to_index)

    node_positions = {}
    for geo_id, data in umst_graph.nodes(data=True):
        idx = tract_to_index[geo_id]
        lat = float(data.get('lat', 0))
        lon = float(data.get('lon', 0))
        node_positions[idx] = (lat, lon)
    print("    ✓ Node position dictionary built.")

    # --- 6. Instantiate the DeliveryList ---
    # (This section is the same as before)
    print("\n[3] Generating all delivery tasks for the simulation...")
    random.seed(42) # Use a fixed seed for reproducibility
    np.random.seed(42)

    deliverylist = DeliveryList(
        max_delivery_time=1800, # all random deliveries will be under this
        load= 400,
        time_tolerance_factor=2.0,
        hours=1,
        peaks=[0.25, 0.75],
        sigma=10,
        num_hotspots=num_hotspots,
        adjacency_matrix=adjacency_matrix,
        node_positions=node_positions
    )

except Exception as e:
    print(f"\n--- !! ERROR !! ---")
    print(f"Could not load graph data or build DeliveryList.")
    print(f"Make sure your Google Drive is mounted and the path is correct:")
    print(f"PATH: {data_dir}")
    print(f"Error: {e}")


[1] Loading graph and census data...
    ✓ Census tracts loaded: 26 tracts (hotspots)
    ✓ UMST Graph loaded: 26 nodes, 47 edges

[2] Building node mappings and adjacency...
✓ Adjacency built: 26 nodes with 47 edges
    ✓ Node position dictionary built.

[3] Generating all delivery tasks for the simulation...
📊 Generating temporal distribution...
🗺️  Pre-computing all shortest paths...


Pre-computing paths: 100%|██████████| 650/650 [00:00<00:00, 33271.06it/s]


✓ Pre-computed 650 shortest paths
✓ Found 636 valid delivery pairs (within 1800s limit)
📦 Generating deliveries with pre-computed paths...


Creating deliveries: 100%|██████████| 18498/18498 [00:00<00:00, 194822.32it/s]

⏱️  Calculating time limits...
✅ Generated 18498 deliveries


In [41]:
import random

def create_delivery_subsets(full_deliverylist, train_frac=0.15, seed=42):
    """
    Splits DeliveryList.deliveries into train/test subsets (by IDs).
    Returns (train_list, test_list) of Delivery objects.
    """
    random.seed(seed)
    all_deliveries = full_deliverylist.deliveries
    n_total = len(all_deliveries)
    n_train = int(train_frac * n_total)

    indices = list(range(n_total))
    random.shuffle(indices)
    train_idx = set(indices[:n_train])

    train_deliveries = [all_deliveries[i] for i in train_idx]
    test_deliveries  = [all_deliveries[i] for i in range(n_total) if i not in train_idx]

    print(f"✓ Split deliveries: {len(train_deliveries)} train, {len(test_deliveries)} test (of {n_total})")
    # Debug sample
    print("  Example train IDs:", [d.id for d in train_deliveries[:5]])
    print("  Example test  IDs:", [d.id for d in test_deliveries[:5]])
    return train_deliveries, test_deliveries

train_deliveries, test_deliveries = create_delivery_subsets(deliverylist, train_frac=0.15)

✓ Split deliveries: 2774 train, 15724 test (of 18498)
  Example train IDs: [8194, 7, 8200, 8203, 8205]
  Example test  IDs: [0, 1, 2, 3, 4]


In [42]:
class DeliveryListSubset:
    """Wraps a subset of Delivery objects for quick experiments."""
    def __init__(self, deliveries, parent_deliverylist):
        self.deliveries = deliveries
        self.adjacency_matrix = parent_deliverylist.adjacency_matrix
        self.node_positions   = parent_deliverylist.node_positions

    def reset_deliveries(self):
        for d in self.deliveries:
            d.reset()


In [43]:
train_dl = DeliveryListSubset(train_deliveries, deliverylist)
test_dl  = DeliveryListSubset(test_deliveries, deliverylist)
print(f"Train subset has {len(train_dl.deliveries)} deliveries.")

Train subset has 2774 deliveries.


In [50]:
# --- Minimal DeliveryEnvShared + SharedDQNAgent implementations ---
# Drop this cell into your notebook (after Delivery & DeliveryList definitions)

import numpy as np
import random
from collections import deque
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class DeliveryEnvShared:
    """
    Simplified shared-environment for MADDPG-style evaluation used by run_rl_v7().
    - deliverylist: DeliveryList instance (with .deliveries list of Delivery objects)
    - adjacency: adjacency matrix (NxN numpy array or list-of-lists) with edge weights (km)
    - node_positions: dict or list mapping node index -> (x,y) (only for _dist)
    - n_agents: number of vehicles
    - capacity: integer per vehicle
    - sim_duration: maximum sim time (seconds / steps)
    Notes:
      - Actions are discrete integers = target node index.
      - Each step increments internal time by 1 (one step ~ 1s). Use same units as Delivery.start_time.
      - env.agents is a list of dicts with at least a 'node' field (current node).
      - env.reset(seed=...) returns an observation array shape (n_agents, obs_dim).
      - env.step(actions) returns obs, rew, dones, info similar to gym.
    """
    def __init__(self, deliverylist, adjacency, node_positions,
             n_agents=None, capacity=2, sim_duration=4500):
        self.deliverylist = deliverylist

        # --- FIX: handle both dict and numpy adjacency formats safely ---
        if isinstance(adjacency, dict):
            self.adjacency = adjacency
            self.N = len(adjacency)
        else:
            self.adjacency = np.array(adjacency)
            self.N = self.adjacency.shape[0]

        self.node_positions = node_positions
        self.n_agents = n_agents or 1
        self.capacity = capacity
        self.sim_duration = sim_duration

        # Extract neighbors list
        self.neighbors = []
        for i in range(self.N):
            if isinstance(self.adjacency, dict):
                # adjacency[i] -> [(neighbor, dist, time), ...]
                self.neighbors.append([nbr for nbr, _, _ in self.adjacency.get(i, [])])
            else:
                self.neighbors.append(np.where(self.adjacency[i] > 0)[0].tolist())

        # Precompute shortest paths
        self._shortest_paths = {}
        self._compute_all_pairs_shortest_paths()

        # Internal state
        self.time = 0
        self.agents = None
        self.deliveries = None
        self._obs_dim = 3  # [node_norm, carried_norm, time_norm]

        print(f"[DEBUG] Environment created with {self.n_agents} agents, "
            f"{len(deliverylist.deliveries)} deliveries, {self.N} nodes.")


    def _compute_all_pairs_shortest_paths(self):
        # BFS from each node (since N likely modest). Store as list of next-hop lists.
        for s in range(self.N):
            # BFS tree
            prev = [-1]*self.N
            q = deque([s])
            prev[s] = s
            while q:
                u = q.popleft()
                for v in self.neighbors[u]:
                    if prev[v] == -1:
                        prev[v] = u
                        q.append(v)
            # recover paths
            for d in range(self.N):
                if prev[d] == -1:
                    continue
                path = []
                cur = d
                while cur != s:
                    path.append(cur)
                    cur = prev[cur]
                path.append(s)
                path.reverse()
                self._shortest_paths[(s,d)] = path

    def _path(self, a, b):
        if a == b:
            return [a]
        return self._shortest_paths.get((a,b), [a])  # fallback to staying put

    def _dist(self, a, b):
        # Euclidean if positions provided, else graph-heuristic using adjacency weights
        try:
            pa = self.node_positions[a]; pb = self.node_positions[b]
            dx = pa[0] - pb[0]; dy = pa[1] - pb[1]
            return math.hypot(dx, dy)
        except Exception:
            # sum along shortest path using adjacency weights (if adjacency contains distances)
            path = self._path(a,b)
            d = 0.0
            for i in range(len(path)-1):
                d += self.adjacency[path[i], path[i+1]]
            return d

    def reset(self, seed=None):
        if seed is not None:
            random.seed(seed); np.random.seed(seed)
        # Reset global deliveries to their initial state if method exists
        try:
            self.deliverylist.reset_deliveries()
        except Exception:
            pass
        self.deliveries = self.deliverylist.deliveries

        # Place agents: start at randomly sampled nodes (or at 0 if deterministic)
        # Choose nodes that exist in node_positions
        start_nodes = list(range(self.N))
        if len(start_nodes) >= self.n_agents:
            starts = random.sample(start_nodes, self.n_agents)
        else:
            starts = [start_nodes[i % len(start_nodes)] for i in range(self.n_agents)]

        self.agents = []
        for a in range(self.n_agents):
            agent = {
                'id': a,
                'node': int(starts[a]),
                'target': int(starts[a]),
                'carried': [],  # list of delivery ids currently on this vehicle
                'distance_travelled': 0.0,
            }
            self.agents.append(agent)

        self.time = 0

        # initial observation: one row per agent
        obs = np.zeros((self.n_agents, self._obs_dim), dtype=float)
        for i,ag in enumerate(self.agents):
            obs[i] = self._agent_obs(i)
        return obs

    def _agent_obs(self, agent_idx):
        ag = self.agents[agent_idx]
        node_norm = ag['node'] / max(1, self.N-1)
        carried = len(ag['carried']) / max(1, self.capacity)
        time_norm = self.time / max(1, self.sim_duration)
        return np.array([node_norm, carried, time_norm], dtype=float)

    def step(self, actions):
        """
        actions: array-like of ints of length n_agents (target node indices)
        Returns: obs (n_agents, obs_dim), rewards (n_agents,), dones (n_agents,), info (dict)
        """
        actions = np.asarray(actions, dtype=int)
        # clip action node indices to [0, N-1]
        actions = np.clip(actions, 0, self.N-1)

        pickups = 0
        deliveries_done = 0
        reward = np.zeros(self.n_agents, dtype=float)

        # For each agent: set proposed target
        for i,a in enumerate(self.agents):
            a['target'] = int(actions[i])

        # Move each agent one graph hop towards its target (if not already there)
        for a in self.agents:
            cur = a['node']
            tgt = a['target']
            if cur != tgt:
                path = self._path(cur, tgt)
                # path[0] == cur, go to next hop if exists
                if len(path) >= 2:
                    nxt = path[1]
                else:
                    nxt = cur
                # update distance using adjacency weights or euclid
                d = self._dist(cur, nxt)
                a['distance_travelled'] += d
                a['node'] = int(nxt)

        # Update time
        self.time += 1
        if self.time > self.sim_duration:
            done_global = True
        else:
            done_global = False

        # Pickup logic: if agent at a delivery start_node whose start_time <= current time and not picked
        for a in self.agents:
            if len(a['carried']) >= self.capacity:
                continue
            # find deliveries at this node available for pickup
            for d in self.deliveries:
                if (not d.in_transition) and (not d.completed) and (d.start_node == a['node']):
                    if self.time >= d.start_time:
                        # pick it up
                        d.in_transition = True
                        d.current_node = a['node']
                        d.path_index = 0
                        a['carried'].append(d.id)
                        d.current_bundle_id = a['id']
                        pickups += 1
                        # small positive reward for pickup
                        reward[a['id']] += 1.0
                        if len(a['carried']) >= self.capacity:
                            break

        # Move deliveries that are in_transition: advance them if their assigned vehicle is at next node
        for d in self.deliveries:
            if d.in_transition and (not d.completed):
                # find the vehicle carrying it (if assigned)
                carrier = None
                for a in self.agents:
                    if d.id in a['carried']:
                        carrier = a
                        break
                if carrier is None:
                    # no carrier (shouldn't happen); leave waiting
                    continue
                # if carrier at end node -> deliver
                if carrier['node'] == d.end_node:
                    d.completed = True
                    d.successful = True if (self.time <= d.time_limit if d.time_limit else True) else False
                    d.end_time = self.time
                    d.in_transition = False
                    d.current_node = d.end_node
                    deliveries_done += 1
                    # remove from carrier
                    if d.id in carrier['carried']:
                        carrier['carried'].remove(d.id)
                    # reward for delivery: bonus proportional to on-time
                    reward[carrier['id']] += 5.0 if d.successful else 1.0
                    d.distance_traveled = self._dist(d.start_node, d.end_node)
                else:
                    # delivery follows carrier node for book-keeping
                    d.current_node = carrier['node']

        # compute vehicle distance total (for info)
        total_vehicle_distance = sum(a['distance_travelled'] for a in self.agents)

        # observations
        obs = np.zeros((self.n_agents, self._obs_dim), dtype=float)
        for i,_ in enumerate(self.agents):
            obs[i] = self._agent_obs(i)

        # dones per-agent: end when sim time exceeded
        dones = np.array([done_global]*self.n_agents)

        info = {
            'pickups': pickups,
            'deliveries': deliveries_done,
            'vehicle_distance': total_vehicle_distance
        }

        # rewards: return as numpy array for compatibility
        return obs, reward, dones, info

# --------------------
# Shared minimal DQN-style policy wrapper (vectorized)
# --------------------
class SimpleQNet(nn.Module):
    def __init__(self, obs_dim, n_actions, hidden=64):
        super().__init__()
        self.linear1 = nn.Linear(obs_dim, hidden)
        self.linear2 = nn.Linear(hidden, hidden)
        self.head = nn.Linear(hidden, n_actions)

    def forward(self, x):
        x = F.relu(self.linear1(x))
        x = F.relu(self.linear2(x))
        return self.head(x)

class SharedDQNAgent:
    """
    Minimal shared DQN-style agent that exposes `act(obs_batch, epsilon=0.0)`.
    - obs_batch: numpy array (B, obs_dim) or torch tensor
    - n_actions: typically set to number of nodes (agent chooses which node to go to)
    - This class does NOT implement training loops or replay buffer here;
      it is only an evaluation-time policy wrapper.
    """
    def __init__(self, obs_dim, n_actions, device=None, hidden=64):
        self.obs_dim = obs_dim
        self.n_actions = n_actions
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.net = SimpleQNet(obs_dim, n_actions, hidden).to(self.device)
        # random init; user can load state_dict later if they have pretrained weights
        self.net.eval()

    def act(self, obs_batch, epsilon=0.0):
        """
        obs_batch: numpy array shape (B, obs_dim)
        returns: numpy array of ints shape (B,) with chosen action indices
        """
        if isinstance(obs_batch, np.ndarray):
            x = torch.from_numpy(obs_batch).float().to(self.device)
        else:
            x = obs_batch.float().to(self.device)

        # epsilon-greedy
        if epsilon > 0.0 and np.random.rand() < epsilon:
            # random actions
            B = x.shape[0]
            return np.random.randint(0, self.n_actions, size=B, dtype=int)

        with torch.no_grad():
            q = self.net(x)  # (B, n_actions)
            actions = q.argmax(dim=1).cpu().numpy().astype(int)
        return actions

    def predict(self, obs_batch):
        "Return Q-values for diagnostic purposes."
        if isinstance(obs_batch, np.ndarray):
            x = torch.from_numpy(obs_batch).float().to(self.device)
        else:
            x = obs_batch.float().to(self.device)
        with torch.no_grad():
            q = self.net(x).cpu().numpy()
        return q

    def load(self, path):
        "Load state_dict (torch) into the internal net."
        st = torch.load(path, map_location=self.device)
        self.net.load_state_dict(st)
        self.net.eval()


In [51]:
env = DeliveryEnvShared(train_dl, adjacency_matrix, node_positions, n_agents=4, capacity=2)
obs = env.reset(seed=123)
print("Initial observation shape:", obs.shape)
print("Sample observation:", obs[0])


[DEBUG] Environment created with 4 agents, 2774 deliveries, 26 nodes.
Initial observation shape: (4, 3)
Sample observation: [0.04 0.   0.  ]


In [52]:
import time

start = time.time()
obs = env.reset(seed=0)
done = np.array([False]*env.n_agents)
steps = 0
while not done.all() and steps < 200:
    actions = np.random.randint(0, env.N, size=env.n_agents)
    obs, rew, done, info = env.step(actions)
    steps += 1
end = time.time()

print(f"Episode finished in {steps} steps, took {end-start:.2f}s.")
print("Info summary:", info)


Episode finished in 200 steps, took 0.08s.
Info summary: {'pickups': 0, 'deliveries': 0, 'vehicle_distance': 7.846275480084915}


In [56]:
# ===== MADDPG training for your DeliveryEnvShared =====
# Paste this cell after you've created train_dl and DeliveryEnvShared (env)
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import os
from collections import deque, namedtuple

# ---------- Hyperparameters (tweakable) ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_AGENTS = 4                     # must match your env instantiation
OBS_DIM = env._obs_dim             # from your env (3)
N_NODES = env.N                    # number of discrete actions (nodes)
ACT_DIM = 1                        # continuous scalar per agent (we'll map -> node index)
HIDDEN = 128
BUFFER_SIZE = 200000
BATCH_SIZE = 256
GAMMA = 0.99
TAU = 0.01
# new local movement hyperparam
LOCAL_MAX_FACTOR = 1.5   # allow neighbors whose edge distance <= LOCAL_MAX_FACTOR * min_edge_distance
LR_ACTOR = 1e-3
LR_CRITIC = 1e-3
MAX_EPISODES = 1500               # start moderate; increase if you have time
MAX_STEPS = 900                    # per episode truncation (you used 200 earlier)
START_TRAIN_AFTER = 2000           # number of transitions before learning begins
TRAIN_EVERY = 2                    # learn every N environment steps
N_UPDATES = 1                      # gradient steps per learning
NOISE_SCALE = 0.6                  # initial exploration noise
NOISE_DECAY = 0.995
MIN_NOISE = 0.05

SAVE_DIR = output_dir
os.makedirs(SAVE_DIR, exist_ok=True)

# ---------- Replay buffer (joint experience) ----------
Transition = namedtuple("Transition", ["obs", "actions", "rewards", "next_obs", "dones"])

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    def push(self, *args):
        self.buffer.append(Transition(*args))
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        # Convert to arrays with shapes: (B, num_agents, obs_dim / 1 etc.)
        obs = np.stack([b.obs for b in batch], axis=0)               # (B, A, obs_dim)
        actions = np.stack([b.actions for b in batch], axis=0)       # (B, A)
        rewards = np.stack([b.rewards for b in batch], axis=0)       # (B, A)
        next_obs = np.stack([b.next_obs for b in batch], axis=0)     # (B, A, obs_dim)
        dones = np.stack([b.dones for b in batch], axis=0)           # (B, A)
        return obs, actions, rewards, next_obs, dones
    def __len__(self):
        return len(self.buffer)

# ---------- Networks ----------
def mlp(in_dim, out_dim, hidden=HIDDEN):
    return nn.Sequential(
        nn.Linear(in_dim, hidden),
        nn.ReLU(),
        nn.Linear(hidden, hidden),
        nn.ReLU(),
        nn.Linear(hidden, out_dim)
    )

class Actor(nn.Module):
    def __init__(self, obs_dim, hidden=HIDDEN):
        super().__init__()
        self.net = mlp(obs_dim, ACT_DIM, hidden)
    def forward(self, x):
        # outputs a continuous scalar (we'll scale to [0, N_NODES-1])
        return self.net(x)

class Critic(nn.Module):
    def __init__(self, joint_obs_dim, joint_act_dim, hidden=HIDDEN):
        super().__init__()
        self.net = mlp(joint_obs_dim + joint_act_dim, 1, hidden)
    def forward(self, joint_obs, joint_actions):
        # joint_obs: (B, A*obs_dim), joint_actions: (B, A*ACT_DIM)
        x = torch.cat([joint_obs, joint_actions], dim=-1)
        return self.net(x).squeeze(-1)  # (B,)

# ---------- Helper mapping continuous -> discrete (node index) ----------
def cont_to_node_index(x_cont):
    # x_cont is float or numpy array scalar(s). We round to nearest integer and clip.
    val = np.round(x_cont).astype(int)
    val = np.clip(val, 0, N_NODES-1)
    return int(val)

def batch_cont_to_nodes(x_cont_batch):
    # x_cont_batch shape (B, A) where each value continuous
    vals = np.round(x_cont_batch).astype(int)
    vals = np.clip(vals, 0, N_NODES-1)
    return vals  # shape (B, A)

# ---------- MADDPG agent containers ----------
class MADDPG:
    def __init__(self, num_agents, obs_dim, act_dim):
        self.num_agents = num_agents
        self.obs_dim = obs_dim
        self.act_dim = act_dim

        # per-agent actor and target
        self.actors = [Actor(obs_dim).to(device) for _ in range(num_agents)]
        self.targets_actor = [Actor(obs_dim).to(device) for _ in range(num_agents)]
        for a, t in zip(self.actors, self.targets_actor):
            t.load_state_dict(a.state_dict())

        # per-agent critic and target (each critic observes joint obs+joint actions)
        joint_obs_dim = num_agents * obs_dim
        joint_act_dim = num_agents * act_dim
        self.critics = [Critic(joint_obs_dim, joint_act_dim).to(device) for _ in range(num_agents)]
        self.targets_critic = [Critic(joint_obs_dim, joint_act_dim).to(device) for _ in range(num_agents)]
        for c, tc in zip(self.critics, self.targets_critic):
            tc.load_state_dict(c.state_dict())

        # optimizers
        self.opt_actors = [optim.Adam(a.parameters(), lr=LR_ACTOR) for a in self.actors]
        self.opt_critics = [optim.Adam(c.parameters(), lr=LR_CRITIC) for c in self.critics]

    def act(self, obs_batch, noise_scale=0.0):
        # obs_batch: np array (B, A, obs_dim) or (A, obs_dim) for single-step
        single = False
        if obs_batch.ndim == 2:  # (A, obs_dim)
            obs_batch = np.expand_dims(obs_batch, axis=0)  # (1, A, obs_dim)
            single = True
        B = obs_batch.shape[0]
        obs_t = torch.tensor(obs_batch, dtype=torch.float32, device=device)  # (B, A, obs_dim)
        actions = []
        for i in range(self.num_agents):
            xi = obs_t[:, i, :]  # (B, obs_dim)
            with torch.no_grad():
                out = self.actors[i](xi).cpu().numpy()  # (B, ACT_DIM)
            # add noise
            out = out + np.random.randn(*out.shape) * noise_scale
            actions.append(out.reshape(B, -1))
        # actions list -> (B, A*ACT_DIM)
        acts = np.concatenate(actions, axis=1)
        # map each agent's scalar to nearest node index
        # reshape to (B, A)
        acts_per_agent = acts.reshape(B, self.num_agents)
        discrete = batch_cont_to_nodes(acts_per_agent)
        if single:
            return discrete[0], acts_per_agent[0]  # (A,), (A,) continuous before rounding
        return discrete, acts_per_agent

    def target_act_cont(self, next_obs_batch):
        # returns continuous actions from target actors (B, A)
        B = next_obs_batch.shape[0] if next_obs_batch.ndim == 3 else 1
        obs_t = torch.tensor(next_obs_batch, dtype=torch.float32, device=device)
        conts = []
        for i in range(self.num_agents):
            xi = obs_t[:, i, :]
            with torch.no_grad():
                out = self.targets_actor[i](xi).cpu().numpy()
            conts.append(out.reshape(B, -1))
        conts = np.concatenate(conts, axis=1)  # (B, A*ACT_DIM)
        return conts.reshape(B, self.num_agents)

    def soft_update(self):
        for i in range(self.num_agents):
            for p, tp in zip(self.actors[i].parameters(), self.targets_actor[i].parameters()):
                tp.data.copy_(tp.data * (1.0 - TAU) + p.data * TAU)
            for p, tp in zip(self.critics[i].parameters(), self.targets_critic[i].parameters()):
                tp.data.copy_(tp.data * (1.0 - TAU) + p.data * TAU)

# ---------- Initialize MADDPG and replay ----------
maddpg = MADDPG(NUM_AGENTS, OBS_DIM, ACT_DIM)
replay = ReplayBuffer(BUFFER_SIZE)

# ---------- Utility: convert numpy batch to torch joint tensors ----------
def to_joint_tensors(obs_batch, cont_actions_batch):
    # obs_batch: (B, A, obs_dim) -> flatten to (B, A*obs_dim)
    B = obs_batch.shape[0]
    jobs = torch.tensor(obs_batch.reshape(B, -1), dtype=torch.float32, device=device)
    jacts = torch.tensor(cont_actions_batch.reshape(B, -1), dtype=torch.float32, device=device)
    return jobs, jacts

# ---------- Training loop ----------
total_steps = 0
noise_scale = NOISE_SCALE

print("Starting MADDPG training on train_dl ...")
for ep in range(1, MAX_EPISODES + 1):
    # create fresh env for each episode using train_dl subset (so deliveries reset)
    env_train = DeliveryEnvShared(train_dl, adjacency_matrix, node_positions, n_agents=NUM_AGENTS, capacity=2, sim_duration=MAX_STEPS)
    obs = env_train.reset(seed=ep)  # (A, obs_dim)
    ep_reward = np.zeros(NUM_AGENTS)
    done = np.array([False]*NUM_AGENTS)
    steps = 0

    while (not done.all()) and steps < MAX_STEPS:
        # get discrete actions + continuous pre-round actions
        # get continuous raw actions from maddpg (we'll ignore the global discrete output)
        # maddpg.act returns (discrete, cont_actions) for a single-step; we only need cont_actions here
        _, cont_actions = maddpg.act(obs, noise_scale=noise_scale)  # cont_actions shape (A,)

        # MAP continuous actions -> allowed neighbor moves (local action space)
        # cont_actions: numpy array shape (A,)
        mapped_actions = []
        for i_agent in range(NUM_AGENTS):
            cur_node = env_train.agents[i_agent]['node']
            # adjacency format: dict[cur_node] -> list of (neighbor_index, dist, time)
            neighbors = env_train.adjacency.get(cur_node, [])
            if not neighbors:
                # no outgoing edges: stay put
                mapped_actions.append(cur_node)
                continue

            # compute min edge distance from current node
            min_edge = min(d for (_, d, _) in neighbors)
            # allowed neighbors are those with distance <= LOCAL_MAX_FACTOR * min_edge
            allowed = [nbr for (nbr, d, _) in neighbors if d <= (LOCAL_MAX_FACTOR * min_edge)]

            # fallback: if allowed empty (shouldn't happen), use all neighbors
            if len(allowed) == 0:
                allowed = [nbr for (nbr, _, _) in neighbors]

            # map continuous scalar -> index in allowed list (stable mapping)
            # cont = cont_actions[i_agent] can be any real; map via tanh -> [0,1] then scale
            cont = float(cont_actions[i_agent])
            s = (np.tanh(cont) + 1.0) / 2.0  # in [0,1]
            idx = int(np.clip(round(s * (len(allowed) - 1)), 0, len(allowed) - 1))
            next_node = allowed[idx]
            mapped_actions.append(int(next_node))

        # now step the environment with mapped local node indices
        next_obs, rew, done, info = env_train.step(mapped_actions)

        # store joint experience: we keep cont_actions (continuous) in replay so training stays as-is
        replay.push(obs.copy(), cont_actions.copy(), rew.copy(), next_obs.copy(), done.copy())
        obs = next_obs
        ep_reward += rew
        steps += 1
        total_steps += 1

        # learning
        if len(replay) > BATCH_SIZE and total_steps > START_TRAIN_AFTER and total_steps % TRAIN_EVERY == 0:
            for _ in range(N_UPDATES):
                # sample
                obs_b, acts_b, rews_b, next_obs_b, dones_b = replay.sample(BATCH_SIZE)  # numpy arrays
                # convert to tensors: joint
                jobs, jacts = to_joint_tensors(obs_b, acts_b)
                # flatten next observations and actions correctly
                B = next_obs_b.shape[0]
                jnext_obs = torch.tensor(next_obs_b.reshape(B, -1), dtype=torch.float32, device=device)
                next_cont_actions = maddpg.target_act_cont(next_obs_b)  # (B, A)
                jnext_acts = torch.tensor(next_cont_actions.reshape(B, -1), dtype=torch.float32, device=device)
                # for each agent update critic and actor
                for agent_idx in range(NUM_AGENTS):
                    critic = maddpg.critics[agent_idx]
                    target_critic = maddpg.targets_critic[agent_idx]
                    opt_c = maddpg.opt_critics[agent_idx]

                    # get reward for this agent in batch
                    r = torch.tensor(rews_b[:, agent_idx], dtype=torch.float32, device=device)  # (B,)
                    d = torch.tensor(dones_b[:, agent_idx].astype(float), dtype=torch.float32, device=device)

                    # compute target Q: r + gamma*(1-d)*Q_target(next_joint_obs, next_joint_actions)
                    with torch.no_grad():
                        q_next = target_critic(jnext_obs.reshape(BATCH_SIZE, -1),
                       jnext_acts[:, :].to(device))

                        # q_next shape (B,) ; note: we used same target critic shape dims; if mismatch, ensure shapes align
                        q_target = r + GAMMA * (1.0 - d) * q_next

                    # current Q
                    q_val = critic(jobs, jacts)
                    critic_loss = nn.MSELoss()(q_val, q_target.detach())

                    opt_c.zero_grad()
                    critic_loss.backward()
                    opt_c.step()

                    # Actor update (policy gradient)
                    actor = maddpg.actors[agent_idx]
                    opt_a = maddpg.opt_actors[agent_idx]

                    # compute actions for each agent in batch with current actor (continuous)
                    # we need to replace agent_idx's cont action and keep others fixed from acts_b
                    obs_tensor = torch.tensor(obs_b.reshape(BATCH_SIZE, NUM_AGENTS, OBS_DIM)[:, agent_idx, :], dtype=torch.float32, device=device)
                    # compute agent's cont actions
                    cont_act_pred = actor(obs_tensor)  # (B, ACT_DIM)
                    # build joint actions tensor replacing agent_idx part
                    acts_pred = torch.tensor(acts_b, dtype=torch.float32, device=device)  # (B, A)
                    acts_pred[:, agent_idx] = cont_act_pred.squeeze(-1)
                    jacts_pred = acts_pred.reshape(BATCH_SIZE, -1)
                    # evaluate critic with predicted joint actions
                    q_pred = critic(jobs, jacts_pred)
                    actor_loss = -q_pred.mean()

                    opt_a.zero_grad()
                    actor_loss.backward()
                    opt_a.step()

                # soft update targets
                maddpg.soft_update()

        # end step loop

    # end episode
    noise_scale = max(MIN_NOISE, noise_scale * NOISE_DECAY)
    if ep % 10 == 0 or ep == 1:
        print(f"Episode {ep}/{MAX_EPISODES} | steps {steps} | mean_reward {ep_reward.mean():.2f} | noise {noise_scale:.3f} | replay {len(replay)}")

    if ep % 20 == 0:
        print(f"[DEBUG-EVAL] Ep{ep} pickups={info['pickups']} deliveries={info['deliveries']}")

    # put this inside the episode loop, after mapping (for debugging only)
    if ep <= 3 and steps < 5:
        print(f"[DEBUG-ACT] step{steps} cont={np.round(cont_actions,2)} mapped={mapped_actions}")


    # periodic evaluation on test subset & save model
    if ep % 100 == 0:
        torch.save({
            'actors': [a.state_dict() for a in maddpg.actors],
            'critics': [c.state_dict() for c in maddpg.critics],
        }, os.path.join(SAVE_DIR, f"maddpg_ep{ep}.pt"))
        print(f"Saved checkpoint ep{ep}")

print("Training finished.")
# Save final
torch.save({
    'actors': [a.state_dict() for a in maddpg.actors],
    'critics': [c.state_dict() for c in maddpg.critics],
}, os.path.join(SAVE_DIR, "maddpg_final.pt"))
print("Final models saved.")


Starting MADDPG training on train_dl ...
[DEBUG] Environment created with 4 agents, 2774 deliveries, 26 nodes.
Episode 1/1500 | steps 900 | mean_reward 5.00 | noise 0.597 | replay 900
[DEBUG] Environment created with 4 agents, 2774 deliveries, 26 nodes.
[DEBUG] Environment created with 4 agents, 2774 deliveries, 26 nodes.
[DEBUG] Environment created with 4 agents, 2774 deliveries, 26 nodes.
[DEBUG] Environment created with 4 agents, 2774 deliveries, 26 nodes.
[DEBUG] Environment created with 4 agents, 2774 deliveries, 26 nodes.
[DEBUG] Environment created with 4 agents, 2774 deliveries, 26 nodes.
[DEBUG] Environment created with 4 agents, 2774 deliveries, 26 nodes.
[DEBUG] Environment created with 4 agents, 2774 deliveries, 26 nodes.
[DEBUG] Environment created with 4 agents, 2774 deliveries, 26 nodes.
Episode 10/1500 | steps 900 | mean_reward 2.00 | noise 0.571 | replay 9000
[DEBUG] Environment created with 4 agents, 2774 deliveries, 26 nodes.
[DEBUG] Environment created with 4 agents

KeyboardInterrupt: 